In [9]:
%pip install unidecode openpyxl
%pip install pandas
%pip install plotly.express
%pip install --upgrade nbformat

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [10]:
from unidecode import unidecode
import pandas as pd
df = pd.read_excel ("BASE PNADC 2012 A 2017 ABR2019 10042019.xlsx", engine='openpyxl')

def remove_accents(text):
  return unidecode(str(text))

# Retirando acentos das colunas
df['NOME_AGREGA'] = df['NOME_AGREGA'].apply(remove_accents)
df['SIGLA_AGREGA'] = df['SIGLA_AGREGA'].apply(remove_accents)


# Padronizando o nome de uma coluna com o aumento das letras (ex: Estado OU RM)
df['SIGLA_AGREGA'] = df['SIGLA_AGREGA'].str.upper()
df['NOME_AGREGA'] = df['NOME_AGREGA'].str.upper()

# Selecionando dados a partir de um ano específico
df_filtrado = df[df['Ano'] >= 2014].copy()
df_filtrado

# Removendo linhas com valores NaN na coluna 'RDPC'
df_filtrado.dropna(subset=['RDPC'], inplace=True)

# Criação de um dicionário (de/para) para mapear Estado -> Região
regioes_map = {
    'AC': 'NORTE', 'AM': 'NORTE', 'AP': 'NORTE', 'PA': 'NORTE', 'RO': 'NORTE', 'RR': 'NORTE', 'TO': 'CENTRO-OESTE',
    'AL': 'NORDESTE', 'BA': 'NORDESTE', 'CE': 'NORDESTE', 'MA': 'NORDESTE', 'PB': 'NORDESTE', 'PE': 'NORDESTE',
    'PI': 'NORDESTE', 'RN': 'NORDESTE', 'SE': 'NORDESTE',
    'GO': 'CENTRO-OESTE', 'MS': 'CENTRO-OESTE', 'MT': 'CENTRO-OESTE', 'DF': 'CENTRO-OESTE',
    'ES': 'SUDESTE', 'MG': 'SUDESTE', 'RJ': 'SUDESTE', 'SP': 'SUDESTE',
    'PR': 'SUL', 'RS': 'SUL', 'SC': 'SUL', 'RM MANAUS': 'NORTE', 'RM BELEM': 'NORTE', 'RM MACAPA': 'NORTE',
    'RM SAO LUIS': 'NORTE', 'RM TERESINA': 'NORTE', 'RM FORTALEZA': 'NORDESTE', 'RM NATAL': 'NORTE',
    'RM JOAO PESSOA': 'NORDESTE', 'RM RECIFE': 'NORDESTE', 'RM MACEIO': 'NORDESTE', 'RM ARACAJU': 'NORDESTE','RM SALVADOR': 'NORDESTE',
    'RM BELO HORIZONTE': 'SUDESTE', 'RM VITORIA': 'SUDESTE', 'RM RIO DE JANEIRO': 'SUDESTE', 'RM SAO PAULO': 'SUDESTE',
    'RM CURITIBA': 'SUL', 'RM FLORIANOPOLIS': 'SUL', 'RM CUIABA': 'CENTRO-OESTE', 'RM PORTO ALEGRE': 'SUL', 'RM GOIANIA': 'CENTRO-OESTE', 'BRASIL': 'País'
}

df_filtrado['REG_AGREGA'] = df_filtrado['SIGLA_AGREGA'].map(regioes_map)

# Mapear RMs para seus respectivos estados para agregação por estado
estado_map_for_aggregation = {
    'AC': 'AC', 'AM': 'AM', 'AP': 'AP', 'PA': 'PA', 'RO': 'RO', 'RR': 'RR', 'TO': 'TO',
    'AL': 'AL', 'BA': 'BA', 'CE': 'CE', 'MA': 'MA', 'PB': 'PB', 'PE': 'PE',
    'PI': 'PI', 'RN': 'RN', 'SE': 'SE',
    'GO': 'GO', 'MS': 'MS', 'MT': 'MT', 'DF': 'DF',
    'ES': 'ES', 'MG': 'MG', 'RJ': 'RJ', 'SP': 'SP',
    'PR': 'PR', 'RS': 'RS', 'SC': 'SC',
    'RM MANAUS': 'AM', 'RM BELEM': 'PA', 'RM MACAPA': 'AP',
    'RM SAO LUIS': 'MA', 'RM TERESINA': 'PI', 'RM FORTALEZA': 'CE', 'RM NATAL': 'RN',
    'RM JOAO PESSOA': 'PB', 'RM RECIFE': 'PE', 'RM MACEIO': 'AL', 'RM ARACAJU': 'SE','RM SALVADOR': 'BA',
    'RM BELO HORIZONTE': 'MG', 'RM VITORIA': 'ES', 'RM RIO DE JANEIRO': 'RJ', 'RM SAO PAULO': 'SP',
    'RM CURITIBA': 'PR', 'RM FLORIANOPOLIS': 'SC', 'RM CUIABA': 'MT', 'RM PORTO ALEGRE': 'RS', 'RM GOIANIA': 'GO', 'BRASIL': 'BRASIL'
}

df_filtrado['ESTADO_PARA_AGREGACAO'] = df_filtrado['SIGLA_AGREGA'].map(estado_map_for_aggregation)


# CÁLCULO DO INDICADOR PRINCIPAL (Ex: Valor Original)
indicador_principal = 'RDPC'

# Agregação por Estado (usando a nova coluna para agregação)
df_estado = df_filtrado.groupby('ESTADO_PARA_AGREGACAO')[indicador_principal].first().reset_index()
df_estado.rename(columns={indicador_principal: f'Valor_{indicador_principal}'}, inplace=True)

# Agregação por Região (usando a coluna REG_AGREGA original)
df_regiao = df_filtrado.groupby('REG_AGREGA')[indicador_principal].first().reset_index()
df_regiao.rename(columns={indicador_principal: f'Valor_{indicador_principal}'}, inplace=True)

print(df_filtrado[['SIGLA_AGREGA', 'REG_AGREGA', 'ESTADO_PARA_AGREGACAO', indicador_principal]])
print(df_estado[['ESTADO_PARA_AGREGACAO', f'Valor_{indicador_principal}']])
print(df_regiao[['REG_AGREGA', f'Valor_{indicador_principal}']])

# Salvar o dataframe tratado em um arquivo Excel
df_filtrado.to_excel('dados_pnadc_tratados.xlsx', index=False)

print("Arquivo 'dados_pnadc_tratados.xlsx' criado com sucesso!")

    SIGLA_AGREGA    REG_AGREGA ESTADO_PARA_AGREGACAO     RDPC
4             AC         NORTE                    AC   507.32
5             AC         NORTE                    AC   497.33
10            AL      NORDESTE                    AL   443.96
11            AL      NORDESTE                    AL   426.14
16            AM         NORTE                    AM   503.04
..           ...           ...                   ...      ...
281           SE      NORDESTE                    SE   540.99
286           SP       SUDESTE                    SP  1212.19
287           SP       SUDESTE                    SP  1133.15
292           TO  CENTRO-OESTE                    TO   577.64
293           TO  CENTRO-OESTE                    TO   607.91

[98 rows x 4 columns]
   ESTADO_PARA_AGREGACAO  Valor_RDPC
0                     AC      507.32
1                     AL      443.96
2                     AM      503.04
3                     AP      591.25
4                     BA      523.73
5          

In [11]:
import plotly.express as px


def plot_regiao(dataframe, indicador):
  """Gera e retorna um gráfico de barras da renda per capita por Região."""
  fig_regiao = px.bar(
      dataframe,
      x=f'REG_AGREGA',
      y=f'Valor_{indicador}',
      title=f'Renda per capita por Região e do Brasil',
      color='REG_AGREGA',
      text=f'Valor_{indicador}',
      labels={f'Valor_{indicador}': 'Valor', 'REG_AGREGA': 'Regiões'}
  )
  fig_regiao.update_traces(texttemplate='%{text:.2f}', textposition='outside')
  return fig_regiao

plot_regiao(df_regiao, indicador_principal)

In [12]:

import plotly.express as px


def plot_estado(dataframe, indicador):
  """Gera e retorna um gráfico de barras da renda per capita por Estado."""
  fig_estado = px.bar(
      dataframe,
      x='ESTADO_PARA_AGREGACAO',
      y=f'Valor_{indicador}',
      title=f'Renda per capita por Estado',
      color='ESTADO_PARA_AGREGACAO',
      text=f'Valor_{indicador}',
      labels={f'Valor_{indicador}': 'Valor', 'ESTADO_PARA_AGREGACAO': 'Estado'}
  )
  fig_estado.update_traces(texttemplate='%{text:.2f}', textposition='outside')
  return fig_estado

# Para exibir o gráfico, chame a função na célula:
plot_estado(df_estado, indicador_principal)